<a href="https://colab.research.google.com/github/JBadawood/Mathematical_Optimization/blob/main/KAU-IE616/Case-Study/Sportano.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **IE616 - Linear Programming**
\
# SPORTANO Case Study

Sportano is a famous brand that makes sports equipment such as:
* boots,
* jerseys,
* footballs, and more.

Four different sizes of balls designed for different age groups are made in three manufacturing sites. Each site has labor and machine-hour limitations, and the values for each are given in Tables 1–3.

**Table 1:** Product-Resource Constraints: Site 1

|Football Type|Labor (Hours/Unit)|Machine (Hours/Unit)|Material (Lb./Uint)|
|--|--|--|--|
|Size 2|3|8|1.0|
|Size 3|3|8.5|1.1|
|Size 4|4|9|1.2|
|Size 5|4|9|1.3|
|Total available|6000|10,000|---|

**Table 2:** Product-Resource Constraints: Site 2

|Football Type|Labor (Hours/Unit)|Machine (Hours/Unit)|Material (Lb./Uint)|
|--|--|--|--|
|Size 2|3.5|7|1.1|
|Size 3|3.5|7|1.0|
|Size 4|4.5|8|1.1|
|Size 5|4.5|9|1.4|
|Total available|5000|12,500|---|

**Table 3:** Product-Resource Constraints: Site 3

|Football Type|Labor (Hours/Unit)|Machine (Hours/Unit)|Material (Lb./Uint)|
|--|--|--|--|
|Size 2|3|7.5|1.1|
|Size 3|3.5|7.5|1.1|
|Size 4|4|8.5|1.2=3|
|Size 5|4.5|8.5|1.3|
|Total available|3000|6,000|---|

The process of manufacturing the balls in all production sites is similar, but not quite identical. Hence, each production site uses different amounts of raw material in creating the same ball size. The material requirements for each football are given in the last column of Tables 1–3. The amount of raw material available for the entire production during the planning period is 3,500 pounds.

Sportano has 3 major customers for its products:
* Royal Spanish Football Federation (**RFEF**),
* The Football Association (**TheFA**), and
* The French Football Federation (**FFF**).

The maximum demand for each customer is given in Table 4 for each product.

**Table 4:** Maximum Product Sales

|Football Type|RFEF|TheFA|FFF|
|--|--|--|--|
|Size 2|200|400|200|
|Size 3|300|300|400|
|Size 4|500|200|300|
|Size 5|200|400|300|

Tables 5-7 show:
* the ***selling price*** of each ball size,
* the ***transportation costs***, and
* the ***manufacturing costs*** of each product for each production site.

**Table 5:** Product Sales Price ($) per Unit

|Football Type|RFEF|TheFA|FFF|
|--|--|--|--|
|Size 2|17|16|16|
|Size 3|18|18|17|
|Size 4|22|22|23|
|Size 5|29|26|27|

**Table 6:** Shipping Costs ($) per Unit

|Production Site|RFEF|TheFA|FFF|
|--|--|--|--|
|Site 1|1.0|1.6|1.1|
|Site 2|1.2|1.5|1.0|
|Site 3|1.4|1.5|1.3|

**Table 7:** Production Costs ($) per Unit

|Football Type|Site 1|Site 2|Site 3|
|--|--|--|--|
|Size 2|14|13|14|
|Size 3|16|17|15|
|Size 4|18|20|19|
|Size 5|26|24|23|

Sites 1 and 2 are experiencing a problem with production that is causing an unusual rate of defects in the balls. Based on that, all shipments manufactured in sites 1 and 2 that are used to fulfill the demand for the customers RFEF or TheFA must go through a special inspection to ensure that the balls conform to the standards. This is done by sending the manufactured balls to a central inspection site where they get tested, and then they get sent to their final destination. The capacity of this inspection site is 1,500 balls during one planning period.

Answer the following questions:
1. Determine Sportano’s optimum plan for the production and demand fulfillment.
2.	Find the total cost and revenue associated with the total production for each manufacturing site.
3.	Marketing is trying to convince RFEF to purchase 50% more footballs in a period. Can the current resources in hand accommodate this increase or do we need to add more resources? How much more money can we make if we take on the additional demand?


In [ ]:
!pip install gurobipy

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.4/13.4 MB 26.8 MB/s eta 0:00:00


In [ ]:
from gurobipy import *
from __future__ import division
import numpy as np
import math
import csv

In [ ]:
Sportano_Planning = Model()

Restricted license - for non-production use only - expires 2025-11-24


# **Problem Formulation (Modeling):**

$\displaystyle Maximize \text{ $z$} = \sum_{i=1}^{|I|}\sum_{j=1}^{|J|}\sum_{k=1}^{|K|}(P_{ik}-T_{jk}-C_{ij}) \times x_{ijk}$

\
**Subject to:**

\
$\displaystyle \sum_{i=1}^{|I|}\sum_{k=1}^{|K|}L_{ij} \times x_{ijk} \leq L_j^{max} \text{ for $j \in J$ (for labor hours)}$

\
$\displaystyle \sum_{i=1}^{|I|}\sum_{k=1}^{|K|}M_{ij} \times x_{ijk} \leq M_j^{max} \text{ for $j \in J$ (for machine hours)}$

\
$\displaystyle \sum_{i=1}^{|I|}\sum_{j=1}^{|J|}\sum_{k=1}^{|K|}R_{ij} \times x_{ijk} \leq R^{max} \text{ (for raw material)}$

\
$\displaystyle \sum_{j=1}^{|J|}x_{ijk} \leq D_{ik} \text{ for $i \in I$, $k \in K$ (for demand)}$

\
$\displaystyle \sum_{i=1}^{|I|}\sum_{j=1}^{2}\sum_{k=1}^{2} x_{ijk} \leq I^{max} \text{ (fot inspection)}$

\
$x_{ijk} \geq 0\ \text{ (Non-negativity)}$ *it is defined as the lower bound `lb = 0` at **decision variable** section*

# **Indices:**

$I: \text{ Set of footballs sizes } i\in I$ \
$J: \text{ Set of manufacutring site } j\in J$ \
$K: \text{ Set of customer } k\in K$

$I = \{0, 1, 2, 3 \}$ 0: Football size 2, 1: Football size 3, 2: Football size 4, 3: Football size 5\
$J = \{0, 1, 2 \}$ 0: Production site 1, 1: Production site 2, 2: Production site 3\
$K = \{0, 1, 2 \}$ 0: RFEF, 1:TheFA, 2:FFF


In [ ]:
I = range(4)
J = range(3)
K = range(3)

# **Parameters:**

$L_{ij} \text{: the labor hours requrement for producing a fooball of size $i$ at manufacturing site $j$}$\
$M_{ij} \text{: the machine hours requrement for producing a fooball of size $i$ at manufacturing site $j$}$\
$R_{ij} \text{: the raw materials requrement for producing a fooball of size $i$ at manufacturing site $j$}$

\
$D_{ik} \text{: the maximum demand of a fooball of size $i$ to a customer $k$}$

\
$P_{ik} \text{: the selling price of a fooball of size $i$ to a customer $k$}$\
$T_{jk} \text{: the transportation cost from manufacturing site $j$ to a customer $k$}$\
$C_{ij} \text{: the production cost of a fooball of size $i$ at manufactureing site $j$}$

**Labor Time $(L_{ij})$**

In [ ]:
Labor_Time = {}
for j in J:
  Labor_Time[j] = []

f = open('/content/Labor_Time.csv')
csv_f = csv.reader(f)
for row in csv_f:
  for j in J:
    Labor_Time[j].append(float(row[j]))

In [ ]:
Labor_Time

{0: [3.0, 3.0, 4.0, 4.0], 1: [3.5, 3.5, 4.5, 4.5], 2: [3.0, 3.5, 4.0, 4.5]}

**Machine Time $(M_{ij})$**

In [ ]:
Machine_Time = {}
for j in J:
  Machine_Time[j] = []

f = open('/content/Machine_Time.csv')
csv_f = csv.reader(f)
for row in csv_f:
  for j in J:
    Machine_Time[j].append(float(row[j]))

In [ ]:
Machine_Time

{0: [8.0, 8.5, 9.0, 9.0], 1: [7.0, 7.0, 8.0, 9.0], 2: [7.5, 7.5, 8.5, 8.5]}

**Raw Material $(R_{ij})$**

In [ ]:
Raw_Material = {}
for j in J:
  Raw_Material[j] = []

f = open('/content/Material.csv')
csv_f = csv.reader(f)
for row in csv_f:
  for j in J:
    Raw_Material[j].append(float(row[j]))

In [ ]:
Raw_Material

{0: [1.0, 1.1, 1.2, 1.3], 1: [1.1, 1.0, 1.1, 1.4], 2: [1.1, 1.1, 1.3, 1.3]}

**Maximum Demand $(D_{ik})$**

In [ ]:
Max_Demand = {}
for k in K:
  Max_Demand[k] = []

f = open('/content/Maximum_Demand.csv')
csv_f = csv.reader(f)
for row in csv_f:
  for k in K:
    Max_Demand[k].append(float(row[k]))

In [ ]:
Max_Demand

{0: [200.0, 300.0, 500.0, 200.0],
 1: [400.0, 300.0, 200.0, 400.0],
 2: [200.0, 400.0, 300.0, 300.0]}

**Sales Price $(P_{ik})$**

In [ ]:
Sales_Price = {}
for k in K:
  Sales_Price[k] = []

f = open('/content/Sales_Price.csv')
csv_f = csv.reader(f)
for row in csv_f:
  for k in K:
    Sales_Price[k].append(float(row[k]))

In [ ]:
Sales_Price

{0: [17.0, 18.0, 22.0, 29.0],
 1: [16.0, 18.0, 22.0, 26.0],
 2: [16.0, 17.0, 23.0, 27.0]}

**Transportation Cost $(T_{jk})$**

In [ ]:
Trans_Cost = {}
for k in K:
  Trans_Cost[k] = []

f = open('/content/Shipping_Costs.csv')
csv_f = csv.reader(f)
for row in csv_f:
  for k in K:
    Trans_Cost[k].append(float(row[k]))

In [ ]:
Trans_Cost

{0: [1.0, 1.2, 1.4], 1: [1.6, 1.5, 1.5], 2: [1.1, 1.0, 1.3]}

**Production Cost $(C_{ij})$**

In [ ]:
Prod_Cost = {}
for j in J:
  Prod_Cost[j] = []

f = open('/content/Production_Costs.csv')
csv_f = csv.reader(f)
for row in csv_f:
  for j in J:
    Prod_Cost[j].append(float(row[j]))

In [ ]:
Prod_Cost

{0: [14.0, 16.0, 18.0, 26.0],
 1: [13.0, 17.0, 20.0, 24.0],
 2: [14.0, 15.0, 19.0, 23.0]}

$L_j^{max}$ , $M_j^{max}$, $R^{max}$, $I^{max}$

In [ ]:
Labor_max = [6000, 5000, 3000]
Machine_max = [10000, 12500, 6000]
Material_max = 3500
Inspection_max = 1500

# **Decision Variables:**

$x_{ijk} \text{: the number of footballs of size $i$ produced at manufacturing site $j$ which will go to customer $k$}$

In [ ]:
x = {}
for i in I:
  for j in J:
    for k in K:
      x[i,j,k] = Sportano_Planning.addVar(vtype=GRB.CONTINUOUS, lb = 0, name="x_"+str(i)+str(j)+str(k))

# **Objective Function:**

$\displaystyle Maximize \text{ $z$} = \sum_{i=1}^{|I|}\sum_{j=1}^{|J|}\sum_{k=1}^{|K|}(P_{ik}-T_{jk}-C_{ij}) \times x_{ijk}$

In [ ]:
Total_Sales_Price = 0
Total_Trans_Cost = 0
Total_Prod_Cost = 0

for i in I:
  for j in J:
    for k in K:
      Total_Sales_Price += Sales_Price[k][i] * x[i,j,k]
      Total_Trans_Cost += Trans_Cost[j][k] * x[i,j,k]
      Total_Prod_Cost += Prod_Cost[j][i] * x[i,j,k]

In [ ]:
Sportano_Planning.setObjective(Total_Sales_Price - Total_Trans_Cost - Total_Prod_Cost, GRB.MAXIMIZE)
Sportano_Planning.update()

# **Constraints:**

**Subject to:**

\
$\displaystyle \sum_{i=1}^{|I|}\sum_{k=1}^{|K|}L_{ij} \times x_{ijk} \leq L_j^{max} \text{ for $j \in J$ (for labor hours)}$

\
$\displaystyle \sum_{i=1}^{|I|}\sum_{k=1}^{|K|}M_{ij} \times x_{ijk} \leq M_j^{max} \text{ for $j \in J$ (for machine hours)}$

\
$\displaystyle \sum_{i=1}^{|I|}\sum_{j=1}^{|J|}\sum_{k=1}^{|K|}R_{ij} \times x_{ijk} \leq R^{max} \text{ (for raw material)}$

\
$\displaystyle \sum_{j=1}^{|J|}x_{ijk} \leq D_{ik} \text{ for $i \in I$, $k \in K$ (for demand)}$

\
$\displaystyle \sum_{i=1}^{|I|}\sum_{j=1}^{2}\sum_{k=1}^{2} x_{ijk} \leq I^{max} \text{ (fot inspection)}$

\
$x_{ijk} \geq 0\ \text{ (Non-negativity)}$ *it is above when we define the lower bound at **decision variable** section*

**Constraints of labor hours**\
$\displaystyle \sum_{i=1}^{|I|}\sum_{k=1}^{|K|}L_{ij} \times x_{ijk} \leq L_j^{max} \text{ for $j \in J$}$

In [ ]:
for j in J:
  Sportano_Planning.addConstr(quicksum(quicksum(Labor_Time[j][i] * x[i,j,k] for i in I) for k in K) <= Labor_max[j])

**Constraints of machine hours**\
$\displaystyle \sum_{i=1}^{|I|}\sum_{k=1}^{|K|}M_{ij} \times x_{ijk} \leq M_j^{max} \text{ for $j \in J$}$

In [ ]:
for j in J:
  Sportano_Planning.addConstr(quicksum(quicksum(Machine_Time[j][i] * x[i,j,k] for i in I) for k in K) <= Machine_max[j])

**Constraints of raw material**\
$\displaystyle \sum_{i=1}^{|I|}\sum_{j=1}^{|J|}\sum_{k=1}^{|K|}R_{ij} \times x_{ijk} \leq R^{max}$

In [ ]:
Sportano_Planning.addConstr(quicksum(quicksum(quicksum(Raw_Material[j][i] * x[i,j,k] for i in I) for j in J) for k in K) <= Material_max)

<gurobi.Constr *Awaiting Model Update*>

**Constraints of demand**\
$\displaystyle \sum_{j=1}^{|J|}x_{ijk} \leq D_{ik} \text{ for $i \in I$, $k \in K$}$

In [ ]:
for i in I:
  for k in K:
    Sportano_Planning.addConstr(quicksum(x[i,j,k] for j in J) <= Max_Demand[k][i])

**Constraints of inspection**\
$\displaystyle \sum_{i=1}^{|I|}\sum_{j=1}^{2}\sum_{k=1}^{2} x_{ijk} \leq I^{max}$

In [ ]:
Sportano_Planning.addConstr(quicksum(quicksum(quicksum(Raw_Material[j][i] * x[i,j,k] for i in I) for j in range(2)) for k in range(2)) <= Inspection_max)

<gurobi.Constr *Awaiting Model Update*>

# **Problem Solution:**

In [ ]:
Sportano_Planning.update()
Sportano_Planning.optimize()

Gurobi Optimizer version 11.0.2 build v11.0.2rc0 (linux64 - "Ubuntu 22.04.3 LTS")

CPU model: Intel(R) Xeon(R) CPU @ 2.20GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 1 physical cores, 2 logical processors, using up to 2 threads

Optimize a model with 20 rows, 36 columns and 160 nonzeros
Model fingerprint: 0x44a5ebbe
Coefficient statistics:
  Matrix range     [1e+00, 9e+00]
  Objective range  [4e-01, 5e+00]
  Bounds range     [0e+00, 0e+00]
  RHS range        [2e+02, 1e+04]
Presolve removed 1 rows and 6 columns
Presolve time: 0.01s
Presolved: 19 rows, 30 columns, 132 nonzeros

Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    1.2444444e+04   2.981246e+03   0.000000e+00      0s
      21    7.0693333e+03   0.000000e+00   0.000000e+00      0s

Solved in 21 iterations and 0.02 seconds (0.00 work units)
Optimal objective  7.069333333e+03


In [ ]:
Sportano_Planning.write("Sportano Planning.lp")
Sportano_Planning.write("Sportano Planning.mps")

## **Linear Programming Display**

In [ ]:
Sportano_Planning.display()

Maximize
2.0 x_000 + 0.8000000000000007 x_001 + 0.5999999999999996 x_002
+ 2.4000000000000004 x_010 + 1.5 x_011 + 1.5 x_012 + 1.9000000000000004 x_020 + x_021
+ 0.6999999999999993 x_022 + x_100 + 0.8000000000000007 x_101 +
-0.40000000000000036 x_102 + -0.6000000000000014 x_110 + -0.5 x_111 + -1.5 x_112
+ 1.8999999999999986 x_120 + 2.0 x_121 + 0.6999999999999993 x_122 + 3.0 x_200
+ 2.8000000000000007 x_201 + 3.6000000000000014 x_202 + 0.3999999999999986 x_210
+ 0.5 x_211 + 1.5 x_212 + 1.8999999999999986 x_220 + 2.0 x_221
+ 2.6999999999999993 x_222 + 2.0 x_300 + -1.1999999999999993 x_301 +
-0.3999999999999986 x_302 + 3.3999999999999986 x_310 + 0.5 x_311 + 1.5 x_312
+ 4.899999999999999 x_320 + 2.0 x_321 + 2.6999999999999993 x_322
Subject To
R0: 3.0 x_000 + 3.0 x_001 + 3.0 x_002 + 3.0 x_100 + 3.0 x_101 + 3.0 x_102 + 4.0 x_200 +
 4.0 x_201 + 4.0 x_202 + 4.0 x_300 + 4.0 x_301 + 4.0 x_302 <= 6000
R1: 3.5 x_010 + 3.5 x_011 + 3.5 x_012 + 3.5 x_110 + 3.5 x_111 + 3.5 x_112 + 4.5 x_210 +
 4.5 x_21

<ipython-input-30-1510b6b58146>:1: DeprecationWarning: Model.display() is deprecated
  Sportano_Planning.display()


# **Results and Discussion:**

## **Optimum Plan:**
a.	Determine Sportano’s optimum plan for the production and demand fulfillment.

In [ ]:
print("Profit =", round(Sportano_Planning.objVal,2))

for i in I:
  for j in J:
    for k in K:
      print("x_" + str(i) + str(j) + str(k) + " =", round(x[i,j,k].x,2))

Profit = 7069.33
x_000 = 0.0
x_001 = 0.0
x_002 = 0.0
x_010 = 200.0
x_011 = 400.0
x_012 = 200.0
x_020 = 0.0
x_021 = 0.0
x_022 = 0.0
x_100 = 0.0
x_101 = 0.0
x_102 = 0.0
x_110 = 0.0
x_111 = 0.0
x_112 = 0.0
x_120 = 273.33
x_121 = 300.0
x_122 = 0.0
x_200 = 500.0
x_201 = 200.0
x_202 = 300.0
x_210 = 0.0
x_211 = 0.0
x_212 = 0.0
x_220 = 0.0
x_221 = 0.0
x_222 = 0.0
x_300 = 0.0
x_301 = 0.0
x_302 = 0.0
x_310 = 0.0
x_311 = 0.0
x_312 = 300.0
x_320 = 200.0
x_321 = 0.0
x_322 = 0.0


## **Total Cost & Revenue:**
b.	Find the total cost and revenue associated with the total production for each manufacturing site.

### **Total Sales Price for Each Site:**

In [ ]:
Total_Sales_Site1=0
for i in I:
  for k in K:
    Total_Sales_Site1 += Sales_Price[k][i] * x[i,0,k].x

In [ ]:
Total_Sales_Site1

22300.0

In [ ]:
Total_Sales_Site2=0
for i in I:
  for k in K:
    Total_Sales_Site2 += Sales_Price[k][i] * x[i,1,k].x

In [ ]:
Total_Sales_Site2

21100.0

In [ ]:
Total_Sales_Site3=0
for i in I:
  for k in K:
    Total_Sales_Site3 += Sales_Price[k][i] * x[i,2,k].x

In [ ]:
Total_Sales_Site3

16120.0

### **Total Transportation Cost for Each Site:**

In [ ]:
Total_Trans_Site1=0
for i in I:
  for k in K:
    Total_Trans_Site1 += Trans_Cost[0][k] * x[i,0,k].x

Total_Trans_Site1

1160.0

In [ ]:
Total_Trans_Site2=0
for i in I:
  for k in K:
    Total_Trans_Site2 += Trans_Cost[1][k] * x[i,1,k].x

Total_Trans_Site2

1670.0

In [ ]:
Total_Trans_Site3=0
for i in I:
  for k in K:
    Total_Trans_Site3 += Trans_Cost[2][k] * x[i,2,k].x

Total_Trans_Site3

820.6666666666667

### **Total Production Cost for Each Site:**

In [ ]:
Total_Prod_Site1=0
for i in I:
  for k in K:
    Total_Prod_Site1 += Prod_Cost[0][i] * x[i,0,k].x

Total_Prod_Site1

18000.0

In [ ]:
Total_Prod_Site2=0
for i in I:
  for k in K:
    Total_Prod_Site2 += Prod_Cost[1][i] * x[i,1,k].x

Total_Prod_Site2

17600.0

In [ ]:
Total_Prod_Site3=0
for i in I:
  for k in K:
    Total_Prod_Site3 += Prod_Cost[2][i] * x[i,2,k].x

Total_Prod_Site3

13200.0

In [ ]:
Site1 = Total_Sales_Site1 - (Total_Trans_Site1 + Total_Prod_Site1)
Site1

3140.0

In [ ]:
Site2 = Total_Sales_Site2 - (Total_Trans_Site2 + Total_Prod_Site2)
Site2

1830.0

In [ ]:
Site3 = Total_Sales_Site3 - (Total_Trans_Site3 + Total_Prod_Site3)
Site3

2099.333333333334

In [ ]:
Site1 + Site2 + Site3

7069.333333333334

### **The Total Cost and Revenue for Each Manufacturing Site:**

The total cost and revenue associated with the total production for each manufacturing site

In [ ]:
print("The Total Cost of Site 1:",Total_Trans_Site1 + Total_Prod_Site1)
print("The revenue of site 1:", Total_Sales_Site1)

print("")

print("The Total Cost of Site 2:",Total_Trans_Site2 + Total_Prod_Site2)
print("The revenue of site 2:", Total_Sales_Site2)

print("")

print("The Total Cost of Site 3:",round(Total_Trans_Site3 + Total_Prod_Site3,2))
print("The revenue of site 3:", Total_Sales_Site3)

The Total Cost of Site 1: 19160.0
The revenue of site 1: 22300.0

The Total Cost of Site 2: 19270.0
The revenue of site 2: 21100.0

The Total Cost of Site 3: 14020.67
The revenue of site 3: 16120.0


## **Additional Demand (RFEF):**
c.	Marketing is trying to convince RFEF to purchase 50% more footballs in a period. Can the current resources in hand accommodate this increase or do we need to add more resources? How much more money can we make if we take on the additional demand?


In [ ]:
shadow_price = Sportano_Planning.getAttr(GRB.Attr.Pi)
slack = Sportano_Planning.getAttr(GRB.Attr.Slack)
RHS_Max = Sportano_Planning.getAttr(GRB.Attr.SARHSUp)
RHS_Min = Sportano_Planning.getAttr(GRB.Attr.SARHSLow)

In [ ]:
print(['%.2f'% elem for elem in slack])
for i in range(20):
  print(round(RHS_Min[i],2), "<= Resource_"+str(i), " <=", round(RHS_Max[i],2), "with shadow price", round(shadow_price[i],2))

['2000.00', '850.00', '93.33', '1000.00', '4200.00', '0.00', '109.33', '0.00', '0.00', '0.00', '26.67', '0.00', '400.00', '0.00', '0.00', '0.00', '0.00', '400.00', '0.00', '0.00']
4000.0 <= Resource_0  <= inf with shadow price 0.0
4150.0 <= Resource_1  <= inf with shadow price 0.0
2906.67 <= Resource_2  <= inf with shadow price 0.0
9000.0 <= Resource_3  <= inf with shadow price 0.0
8300.0 <= Resource_4  <= inf with shadow price 0.0
3950.0 <= Resource_5  <= 6200.0 with shadow price 0.25
3390.67 <= Resource_6  <= inf with shadow price 0.0
173.33 <= Resource_7  <= 200.0 with shadow price 1.4
373.33 <= Resource_8  <= 400.0 with shadow price 0.5
0.0 <= Resource_9  <= 299.39 with shadow price 1.5
273.33 <= Resource_10  <= inf with shadow price 0.0
273.33 <= Resource_11  <= 573.33 with shadow price 0.1
0.0 <= Resource_12  <= inf with shadow price 0.0
475.56 <= Resource_13  <= 500.0 with shadow price 1.91
175.56 <= Resource_14  <= 200.0 with shadow price 1.71
0.0 <= Resource_15  <= 391.11 with

In [ ]:
# Our previous profit before updating the Max Demand of RFEF:
old_profit = round(Sportano_Planning.objVal,2)
old_profit

7069.33

In [ ]:
old_constrs = [7,10,13,16]
new_valRHS = [300,450,750,300] # New maximum demand of RFEF with addtional 50%

In [ ]:
# Updating the RHS for Constrains R7,10,13,16 (Max Demand of RFEF):
for i in range(4):
  Sportano_Planning.setAttr("RHS",Sportano_Planning.getConstrs()[old_constrs[i]],new_valRHS[i])

In [ ]:
# Now the model will be updated to get the new objective value & decision variables :
Sportano_Planning.update()
Sportano_Planning.optimize()

Gurobi Optimizer version 11.0.2 build v11.0.2rc0 (linux64 - "Ubuntu 22.04.3 LTS")

CPU model: Intel(R) Xeon(R) CPU @ 2.20GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 1 physical cores, 2 logical processors, using up to 2 threads

Optimize a model with 20 rows, 36 columns and 160 nonzeros
Coefficient statistics:
  Matrix range     [1e+00, 9e+00]
  Objective range  [4e-01, 5e+00]
  Bounds range     [0e+00, 0e+00]
  RHS range        [2e+02, 1e+04]
LP warm-start: use basis
Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    7.9612727e+03   3.727273e+02   0.000000e+00      0s
       2    7.6132929e+03   0.000000e+00   0.000000e+00      0s

Solved in 2 iterations and 0.01 seconds (0.00 work units)
Optimal objective  7.613292929e+03


In [ ]:
Sportano_Planning.display()

Maximize
2.0 x_000 + 0.8000000000000007 x_001 + 0.5999999999999996 x_002
+ 2.4000000000000004 x_010 + 1.5 x_011 + 1.5 x_012 + 1.9000000000000004 x_020 + x_021
+ 0.6999999999999993 x_022 + x_100 + 0.8000000000000007 x_101 +
-0.40000000000000036 x_102 + -0.6000000000000014 x_110 + -0.5 x_111 + -1.5 x_112
+ 1.8999999999999986 x_120 + 2.0 x_121 + 0.6999999999999993 x_122 + 3.0 x_200
+ 2.8000000000000007 x_201 + 3.6000000000000014 x_202 + 0.3999999999999986 x_210
+ 0.5 x_211 + 1.5 x_212 + 1.8999999999999986 x_220 + 2.0 x_221
+ 2.6999999999999993 x_222 + 2.0 x_300 + -1.1999999999999993 x_301 +
-0.3999999999999986 x_302 + 3.3999999999999986 x_310 + 0.5 x_311 + 1.5 x_312
+ 4.899999999999999 x_320 + 2.0 x_321 + 2.6999999999999993 x_322
Subject To
R0: 3.0 x_000 + 3.0 x_001 + 3.0 x_002 + 3.0 x_100 + 3.0 x_101 + 3.0 x_102 + 4.0 x_200 +
 4.0 x_201 + 4.0 x_202 + 4.0 x_300 + 4.0 x_301 + 4.0 x_302 <= 6000
R1: 3.5 x_010 + 3.5 x_011 + 3.5 x_012 + 3.5 x_110 + 3.5 x_111 + 3.5 x_112 + 4.5 x_210 +
 4.5 x_21

<ipython-input-55-1510b6b58146>:1: DeprecationWarning: Model.display() is deprecated
  Sportano_Planning.display()


In [ ]:
new_profit = round(Sportano_Planning.objVal,2)

In [ ]:
print("Profit =", round(Sportano_Planning.objVal,2))

for i in I:
  for j in J:
    for k in K:
      print("x_" + str(i) + str(j) + str(k) + " =", round(x[i,j,k].x,2))

Profit = 7613.29
x_000 = 0.0
x_001 = 0.0
x_002 = 0.0
x_010 = 300.0
x_011 = 178.79
x_012 = 200.0
x_020 = 0.0
x_021 = 0.0
x_022 = 0.0
x_100 = 0.0
x_101 = 0.0
x_102 = 0.0
x_110 = 0.0
x_111 = 0.0
x_112 = 0.0
x_120 = 160.0
x_121 = 300.0
x_122 = 0.0
x_200 = 750.0
x_201 = 61.11
x_202 = 300.0
x_210 = 0.0
x_211 = 0.0
x_212 = 0.0
x_220 = 0.0
x_221 = 0.0
x_222 = 0.0
x_300 = 0.0
x_301 = 0.0
x_302 = 0.0
x_310 = 0.0
x_311 = 0.0
x_312 = 300.0
x_320 = 300.0
x_321 = 0.0
x_322 = 0.0


if we take on the additional demand the profit will be more by:

In [ ]:
print("the additional demand the profit will be more by",new_profit - old_profit)

the additional demand the profit will be more by 543.96


In [ ]:
shadow_price = Sportano_Planning.getAttr(GRB.Attr.Pi)
slack = Sportano_Planning.getAttr(GRB.Attr.Slack)
RHS_Max = Sportano_Planning.getAttr(GRB.Attr.SARHSUp)
RHS_Min = Sportano_Planning.getAttr(GRB.Attr.SARHSLow)

In [ ]:
# Checking the minimum RHS for the resources:
print(['%.2f'% elem for elem in slack])
for i in range(20):
  print(round(RHS_Min[i],2), "<= Resource_"+str(i), " <=", round(RHS_Max[i],2), "with shadow price", round(shadow_price[i],2))

['1555.56', '1274.24', '40.00', '0.00', '5048.48', '0.00', '104.00', '0.00', '221.21', '0.00', '290.00', '0.00', '400.00', '0.00', '138.89', '0.00', '0.00', '400.00', '0.00', '0.00']
4444.44 <= Resource_0  <= inf with shadow price 0.0
3725.76 <= Resource_1  <= inf with shadow price 0.0
2960.0 <= Resource_2  <= inf with shadow price 0.0
9450.0 <= Resource_3  <= 11250.0 with shadow price 0.13
7451.52 <= Resource_4  <= inf with shadow price 0.0
4800.0 <= Resource_5  <= 6085.71 with shadow price 0.25
3396.0 <= Resource_6  <= inf with shadow price 0.0
78.79 <= Resource_7  <= 478.79 with shadow price 0.9
178.79 <= Resource_8  <= inf with shadow price 0.0
0.0 <= Resource_9  <= 294.55 with shadow price 1.5
160.0 <= Resource_10  <= inf with shadow price 0.0
10.0 <= Resource_11  <= 460.0 with shadow price 0.1
0.0 <= Resource_12  <= inf with shadow price 0.0
611.11 <= Resource_13  <= 811.11 with shadow price 0.2
61.11 <= Resource_14  <= inf with shadow price 0.0
161.11 <= Resource_15  <= 361.11 w

The new approach showed no violation to the optimiality. So, *the current resources in hand accommodate this increase*

# Authors
* Abdulrahman Alharthi
* Fahad Alharbi
* Jameel Badawood
* Moayed Alshaye